
# From Network Flows to Behavioral Graphs
## Straightforward Controlled CG-vs-BDG GAT Pipeline — BCCC-CIC-IDS-2017

This notebook implements a direct comparison between the **Communication Graph (CG)** and the proposed **Behavioral Dependency Graph (BDG)**.

## Communication Graph (CG)

Within the same temporal window:

1. each network flow is a node;
2. flows are connected when they share an exact endpoint;
3. every CG edge has the same scalar edge attribute:

\[
e_{ij}^{CG}=1
\]

```text
Shared endpoint
      ↓
Uniform edge attribute
      ↓
GAT
```

## Behavioral Dependency Graph (BDG)

BDG starts from the **same shared-endpoint edges** and enriches each relationship using:

- communication relation \(C\);
- service relation \(S\);
- protocol relation \(P\);
- traffic-pattern relation \(B\).

Because each candidate edge already shares an endpoint:

\[
C_{ij}=1
\]

The behavioral edge attribute is defined directly as:

\[
\boxed{
e_{ij}^{BDG}
=
\frac{C_{ij}+S_{ij}+P_{ij}+B_{ij}}{4}
=
\frac{1+S_{ij}+P_{ij}+B_{ij}}{4}
}
\]

```text
Shared endpoint
      +
Service
      +
Protocol
      +
Traffic pattern
      ↓
Behavioral edge attribute
      ↓
GAT
```

There is no additional BDG-specific weighting calibration, threshold search, component-selection ablation, or edge filtering.

## Strict controlled comparison

CG and BDG use the same:

- dataset;
- 70/20/10 graph-isolated split;
- nodes;
- shared-endpoint topology;
- node features;
- GAT architecture;
- training settings;
- class weighting;
- random seeds.

They differ only in edge representation:

- CG: uniform edge attribute `1.0`;
- BDG: `(1 + S + P + B) / 4`.

## Only ablation study

Temporal-window size is the only ablation and is performed for both methods:

- 10 seconds
- 30 seconds
- 60 seconds
- 300 seconds
- 600 seconds

One common temporal window is selected using validation Macro-F1 and is then locked before the final multi-seed test.


## Additional non-graph baselines

After the common temporal window is selected and locked using **CG-GAT and CC-BDG-GAT only**, the final locked train/validation/test flow splits are also used to evaluate:

- **** — flow-level 1D convolutional neural network;
- **LightGBM** — flow-level tree-based baseline.

These baselines do **not** participate in temporal-window selection. They are evaluated only after the common graph window has been locked, preventing them from influencing the proposed graph-design ablation.

For fairness:

- LightGBM use the same locked train/validation/test flow rows as CG-GAT and CC-BDG-GAT;
- the same final flow feature columns are used;
- preprocessing is fit on training data only;
- all four methods report the same aggregate metrics:
  accuracy, macro precision, macro recall, macro F1, weighted precision, weighted recall, and weighted F1;
- all four methods are evaluated over the same final seeds;
- per-class precision, recall, F1 and support are saved for every seed;
- a multi-seed per-class mean/std table is produced for all four methods.


In [ ]:

# CELL 1 — INSTALL DEPENDENCIES
!pip -q install torch-geometric scikit-learn pandas numpy matplotlib lightgbm

print("Dependencies installed.")

In [ ]:

# CELL 2 — IMPORTS AND REPRODUCIBILITY
import os, gc, json, math, time, random, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.neighbors import NearestNeighbors
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv

warnings.filterwarnings("ignore")

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [ ]:
# CELL 3 — GOOGLE DRIVE + CONFIGURATION
from google.colab import drive
drive.mount("/content/drive")

# EDIT THIS PATH
DATASET_DIR = Path("/content/drive/MyDrive/BCCC-CIC-IDS-2017")
RESULT_DIR = Path("/content/drive/MyDrive/GAT_RESULT4-1")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

FILE_LABEL_MAP = {
    "botnet_ares.csv": "Botnet_ARES",
    "ddos_loit.csv": "DDoS_LOIT",
    "dos_golden_eye.csv": "DoS_GoldenEye",
    "dos_hulk.csv": "DoS_Hulk",
    "dos_slowhttptest.csv": "DoS_SlowHTTPTest",
    "dos_slowloris.csv": "DoS_Slowloris",
    "ftp_patator.csv": "FTP_Patator",
    "portscan.csv": "PortScan",
    "ssh_patator-new.csv": "SSH_Patator",
    "web_brute_force.csv": "Web_Brute_Force",
    "web_xss.csv": "Web_XSS",
    "monday_benign.csv": "Benign",
    "wednesday_benign.csv": "Benign",
    "friday_benign.csv": "Benign",
}

SELECTED_FILES = list(FILE_LABEL_MAP.keys())
EXCLUDED_FILES = {"heartbleed.csv", "web_sql_injection.csv"}

TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.20, 0.10
WINDOW_SECONDS = [10, 30, 60, 300, 600]

# Communication Graph sparsity.
CG_NEIGHBORS_PER_ENDPOINT = 10


MAX_FLOWS_PER_GRAPH = 5000



ABLATION_EPOCHS = 20
FINAL_EPOCHS = 50
PATIENCE = 8
BATCH_SIZE = 16
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
GAT_HIDDEN_CHANNELS = 16
GAT_HEADS = 4
GAT_SECOND_HEADS = 1
GAT_DROPOUT = 0.30
GAT_ATTENTION_DROPOUT = 0.15
USE_CLASS_WEIGHTS = True

FINAL_SEEDS = [42, 52, 62, 72, 82, 92, 102, 112, 122, 132]

# Flow-level LightGBM baseline configuration.
# This baseline is evaluated only after the common graph window is locked.

LIGHTGBM_N_ESTIMATORS = 800
LIGHTGBM_LEARNING_RATE = 0.05
LIGHTGBM_NUM_LEAVES = 31
LIGHTGBM_SUBSAMPLE = 0.80
LIGHTGBM_COLSAMPLE = 0.80
LIGHTGBM_EARLY_STOPPING_ROUNDS = 50

# Set to an integer for debugging; leave None for the complete experiment.
DEV_MAX_ROWS_PER_FILE = None

print("Dataset:", DATASET_DIR)
print("Results:", RESULT_DIR)
print("Selected files:", len(SELECTED_FILES))







# GAT architecture ablation
ARCHITECTURE_CANDIDATES = [
    "current_2gat_linear",
    "residual_2gat_mlp",
    "shallow_1gat_mlp",
]
ARCH_ABLATION_SEED = 42
ARCH_MLP_HIDDEN = 64
ARCH_MLP_DROPOUT = 0.20



## Leakage policy

The notebook separates three kinds of variables:

**Graph-construction metadata only:**  
`flow_id`, `timestamp`, `src_ip`, `src_port`, `dst_ip`, `dst_port`, `protocol`

**Model/node features:**  
cleaned statistical flow characteristics such as duration, packets, payload bytes, rates, TCP flags, IAT, active/idle measures, etc.

**Target only:**  
`label`

Raw IP addresses, flow IDs and absolute timestamps never enter the GAT feature matrix.


In [ ]:

# CELL 4 — LOAD AND HARMONIZE SELECTED FILES
META_COLUMNS = [
    "flow_id", "timestamp", "src_ip", "src_port",
    "dst_ip", "dst_port", "protocol"
]
TARGET_COLUMN = "label"

frames = []
file_audit = []

for filename in SELECTED_FILES:
    path = DATASET_DIR / filename
    if not path.exists():
        print("[WARNING] Missing:", path)
        continue

    df = pd.read_csv(path, nrows=DEV_MAX_ROWS_PER_FILE, low_memory=False)
    df.columns = [str(c).strip().lower() for c in df.columns]

    required = set(META_COLUMNS + [TARGET_COLUMN])
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"{filename} missing required columns: {missing}")

    original_counts = df[TARGET_COLUMN].astype(str).value_counts(dropna=False).to_dict()

    df["original_label"] = df[TARGET_COLUMN].astype(str)
    df[TARGET_COLUMN] = FILE_LABEL_MAP[filename]
    df["source_file"] = filename

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    bad_ts = int(df["timestamp"].isna().sum())
    df = df.dropna(subset=["timestamp"]).copy()

    file_audit.append({
        "file": filename,
        "assigned_label": FILE_LABEL_MAP[filename],
        "rows_loaded": len(df),
        "bad_timestamp_rows_removed": bad_ts,
        "original_labels_seen": json.dumps(original_counts),
    })
    frames.append(df)

if not frames:
    raise RuntimeError("No files loaded. Check DATASET_DIR.")

data = pd.concat(frames, ignore_index=True)
data["_row_id"] = np.arange(len(data), dtype=np.int64)

file_audit_df = pd.DataFrame(file_audit)
display(file_audit_df)
file_audit_df.to_csv(RESULT_DIR / "01_file_audit.csv", index=False)

print("Combined shape:", data.shape)

In [ ]:

# CELL 5 — DATASET AUDIT + LABEL ENCODING
print("Time range:", data["timestamp"].min(), "to", data["timestamp"].max())
print("Unique src IP:", data["src_ip"].nunique())
print("Unique dst IP:", data["dst_ip"].nunique())
print("Unique src ports:", data["src_port"].nunique())
print("Unique dst ports:", data["dst_port"].nunique())
print("Protocols:", sorted(data["protocol"].astype(str).unique().tolist())[:30])

class_counts = data[TARGET_COLUMN].value_counts().rename_axis("class").reset_index(name="count")
class_counts["percentage"] = class_counts["count"] / len(data) * 100
display(class_counts)
class_counts.to_csv(RESULT_DIR / "02_class_distribution.csv", index=False)

label_encoder = LabelEncoder()
data["y"] = label_encoder.fit_transform(data[TARGET_COLUMN])

CLASS_NAMES = label_encoder.classes_.tolist()
NUM_CLASSES = len(CLASS_NAMES)

print("Number of classes:", NUM_CLASSES)
for i, c in enumerate(CLASS_NAMES):
    print(i, c)

with open(RESULT_DIR / "class_mapping.json", "w") as f:
    json.dump({i: c for i, c in enumerate(CLASS_NAMES)}, f, indent=2)


# Graph-window assignment

Each temporal window is one graph snapshot. If a very dense time window exceeds the configured node safety cap, it is split chronologically into smaller graph chunks.

The train/validation/test split is performed at the **graph-window level**, so nodes in one graph never appear in two different splits.



## Critical split-before-graph rule

The temporal windows are identified first so each flow belongs to exactly one candidate graph window.  
Those **window IDs are then split into 70% Train / 20% Validation / 10% Test before any graph edges are constructed**.

After the split is locked, graph construction is performed independently inside each split:

| Split | Communication Graph | Behavioral Dependency Graph |
|---|---|---|
| Train | built only from Train flows | built only from Train flows |
| Validation | built only from Validation flows | built only from Validation flows |
| Test | built only from Test flows | built only from Test flows |

Therefore, the experiment creates **six separate graph collections**.

There are **no edges between Train, Validation, and Test**, so GAT message passing cannot cross split boundaries.

The Train split alone is used to fit:
- median imputation,
- standardization,
- constant-feature removal,
- duplicate-feature removal,
- BDD traffic-feature normalization.

The fitted preprocessing objects are then applied unchanged to Validation and Test.


In [ ]:

# CELL 6 — WINDOW ASSIGNMENT + 70/20/10 GRAPH SPLIT
def add_window_ids(df, window_seconds, max_flows_per_graph=5000):
    out = df.sort_values("timestamp").copy()

    epoch_s = (out["timestamp"].astype("int64") // 10**9).astype(np.int64)
    out["_window_start_s"] = (epoch_s // window_seconds) * window_seconds
    out["_base_window"] = out["_window_start_s"].astype(str)

    out["_rank_in_window"] = out.groupby("_base_window").cumcount()
    out["_chunk"] = out["_rank_in_window"] // max_flows_per_graph
    out["_graph_id"] = out["_base_window"].astype(str) + "_c" + out["_chunk"].astype(str)

    return out

def graph_summary_table(df):
    base = df.groupby("_graph_id").agg(
        n_flows=("_row_id", "size"),
        window_start=("_window_start_s", "min")
    ).reset_index()

    dominant = df.groupby("_graph_id")["y"].agg(
        lambda s: int(s.value_counts().index[0])
    ).rename("dominant_y").reset_index()

    return base.merge(dominant, on="_graph_id", how="left")

def safe_graph_split(windowed_df, seed=42):
    summary = graph_summary_table(windowed_df)

    graph_ids = summary["_graph_id"].values
    strat = summary["dominant_y"].values

    vc = pd.Series(strat).value_counts()
    strat1 = strat if len(vc) > 1 and vc.min() >= 2 else None

    train_ids, temp_ids = train_test_split(
        graph_ids,
        test_size=VAL_FRAC + TEST_FRAC,
        random_state=seed,
        stratify=strat1
    )

    temp_summary = summary[summary["_graph_id"].isin(temp_ids)].copy()
    temp_strat = temp_summary["dominant_y"].values
    relative_test = TEST_FRAC / (VAL_FRAC + TEST_FRAC)

    vc2 = pd.Series(temp_strat).value_counts()
    strat2 = temp_strat if len(vc2) > 1 and vc2.min() >= 2 else None

    val_ids, test_ids = train_test_split(
        temp_summary["_graph_id"].values,
        test_size=relative_test,
        random_state=seed,
        stratify=strat2
    )

    m = {g: "train" for g in train_ids}
    m.update({g: "val" for g in val_ids})
    m.update({g: "test" for g in test_ids})

    out = windowed_df.copy()
    out["_split"] = out["_graph_id"].map(m)
    return out

# Dataset-size audit for all candidate windows.
rows = []
for ws in WINDOW_SECONDS:
    tmp = add_window_ids(data, ws, MAX_FLOWS_PER_GRAPH)
    s = graph_summary_table(tmp)
    rows.append({
        "window_seconds": ws,
        "graphs": len(s),
        "mean_flows_per_graph": s["n_flows"].mean(),
        "median_flows_per_graph": s["n_flows"].median(),
        "p95_flows_per_graph": s["n_flows"].quantile(.95),
        "max_flows_per_graph": s["n_flows"].max(),
    })

window_flow_audit = pd.DataFrame(rows)
display(window_flow_audit)
window_flow_audit.to_csv(RESULT_DIR / "03_window_flow_audit.csv", index=False)

In [ ]:

# CELL 7 — TRAIN-ONLY FEATURE PREPROCESSING
def numeric_model_features(df):
    excluded = set(META_COLUMNS + [
        TARGET_COLUMN, "original_label", "source_file", "_row_id", "y",
        "_window_start_s", "_base_window", "_rank_in_window",
        "_chunk", "_graph_id", "_split"
    ])
    return [
        c for c in df.columns
        if c not in excluded and pd.api.types.is_numeric_dtype(df[c])
    ]

def exact_duplicate_columns(X):
    duplicate_cols = []
    cols = list(X.columns)
    for i, c in enumerate(cols):
        if c in duplicate_cols:
            continue
        for d in cols[i+1:]:
            if d in duplicate_cols:
                continue
            if X[c].equals(X[d]):
                duplicate_cols.append(d)
    return duplicate_cols

class FlowPreprocessor:
    def fit(self, train_df):
        cols = numeric_model_features(train_df)
        X = train_df[cols].replace([np.inf, -np.inf], np.nan)

        nunique = X.nunique(dropna=False)
        self.constant_cols = nunique[nunique <= 1].index.tolist()
        X = X.drop(columns=self.constant_cols, errors="ignore")

        # Exact duplicate columns. Disable this line if debugging on a huge dataset.
        self.duplicate_cols = exact_duplicate_columns(X)
        X = X.drop(columns=self.duplicate_cols, errors="ignore")

        self.feature_cols = X.columns.tolist()

        self.imputer = SimpleImputer(strategy="median")
        Xi = self.imputer.fit_transform(X)

        self.scaler = StandardScaler()
        self.scaler.fit(Xi)
        return self

    def transform_scaled(self, df):
        X = df[self.feature_cols].replace([np.inf, -np.inf], np.nan)
        Xi = self.imputer.transform(X)
        return self.scaler.transform(Xi).astype(np.float32)

    def transform_imputed(self, df):
        X = df[self.feature_cols].replace([np.inf, -np.inf], np.nan)
        return self.imputer.transform(X).astype(np.float32)

    def report(self):
        return {
            "n_features_final": len(self.feature_cols),
            "constant_cols": self.constant_cols,
            "duplicate_cols": self.duplicate_cols,
            "feature_cols": self.feature_cols,
        }

BDD_TRAFFIC_FEATURES = [
    "duration", "packets_count", "fwd_packets_count", "bwd_packets_count",
    "total_payload_bytes", "fwd_total_payload_bytes", "bwd_total_payload_bytes",
    "bytes_rate", "packets_rate", "fwd_packets_rate", "bwd_packets_rate",
    "down_up_rate", "packets_iat_mean", "packet_iat_std",
    "fwd_packets_iat_mean", "bwd_packets_iat_mean",
    "syn_flag_counts", "ack_flag_counts", "rst_flag_counts"
]

class BDDTrafficPreprocessor:
    def fit(self, train_df):
        self.cols = [c for c in BDD_TRAFFIC_FEATURES if c in train_df.columns]
        if len(self.cols) < 3:
            raise ValueError("Too few BDD traffic features found.")

        X = train_df[self.cols].replace([np.inf, -np.inf], np.nan)
        self.imputer = SimpleImputer(strategy="median")
        Xi = self.imputer.fit_transform(X)
        self.scaler = StandardScaler().fit(Xi)
        return self

    def transform(self, df):
        X = df[self.cols].replace([np.inf, -np.inf], np.nan)
        Z = self.scaler.transform(self.imputer.transform(X)).astype(np.float32)
        norm = np.linalg.norm(Z, axis=1, keepdims=True)
        norm[norm == 0] = 1.0
        return Z / norm


# Behavioral Dependency Graph edge weighting

CG and BDG use the **same shared-endpoint edge set**.

For every shared-endpoint pair:

- Service relation \(S\): same destination port = 1.0; same service family = 0.7; otherwise = 0.0.
- Protocol relation \(P\): same protocol = 1.0; otherwise = 0.0.
- Traffic-pattern relation \(B\): cosine similarity of standardized non-leaky traffic statistics, mapped to \([0,1]\).

The shared-endpoint communication relation is the fixed base term 1.0:

\[
w_{ij}^{BDG}=\frac{1+S_{ij}+P_{ij}+B_{ij}}{4}
\]

No threshold is used. No communication edge is removed.


In [ ]:
# CELL 8 — SHARED-ENDPOINT CANDIDATES + CC-BDG EDGE REPRESENTATIONS

def endpoint_candidate_pairs(gdf, k_per_endpoint=10):
    endpoint_to_indices = defaultdict(list)
    src = gdf["src_ip"].astype(str).to_numpy()
    dst = gdf["dst_ip"].astype(str).to_numpy()
    ts = gdf["timestamp"].to_numpy()

    for idx in range(len(gdf)):
        endpoint_to_indices[src[idx]].append(idx)
        endpoint_to_indices[dst[idx]].append(idx)

    pairs = set()
    for idxs in endpoint_to_indices.values():
        if len(idxs) < 2:
            continue
        idxs = sorted(idxs, key=lambda idx: ts[idx])
        n = len(idxs)
        for pos, a in enumerate(idxs):
            lo = max(0, pos - k_per_endpoint)
            hi = min(n, pos + k_per_endpoint + 1)
            for pos2 in range(lo, hi):
                if pos2 == pos:
                    continue
                b = idxs[pos2]
                i, j = (a, b) if a < b else (b, a)
                if i != j:
                    pairs.add((i, j))
    return pairs


def _service_family_ids(ports):
    numeric = pd.to_numeric(pd.Series(ports), errors="coerce").to_numpy()
    out = np.zeros(len(numeric), dtype=np.int16)
    families = {
        1: {80, 443, 8080, 8443},
        2: {22, 23, 3389},
        3: {20, 21},
        4: {135, 137, 138, 139, 445},
        5: {53},
        6: {67, 68},
        7: {25, 110, 143, 465, 587, 993, 995},
        8: {1433, 1521, 3306, 5432},
    }
    for fam_id, fam_ports in families.items():
        out[np.isin(numeric, list(fam_ports))] = fam_id
    un = out == 0
    out[un & np.isfinite(numeric) & (numeric >= 0) & (numeric <= 1023)] = 9
    out[un & np.isfinite(numeric) & (numeric >= 1024) & (numeric <= 49151)] = 10
    out[un & np.isfinite(numeric) & (numeric > 49151)] = 11
    return out


def compute_bdg_components(gdf, Z, pairs):
    """Return C, S, P, B for each existing shared-endpoint edge."""
    pairs = np.asarray(pairs, dtype=np.int64)
    if len(pairs) == 0:
        empty = np.empty((0,), dtype=np.float32)
        return empty, empty, empty, empty

    i = pairs[:, 0]
    j = pairs[:, 1]

    dport = pd.to_numeric(gdf["dst_port"], errors="coerce").to_numpy()
    proto = gdf["protocol"].astype(str).to_numpy()

    fam = _service_family_ids(dport)
    S = np.zeros(len(pairs), dtype=np.float32)
    same_family = (fam[i] == fam[j]) & (fam[i] != 0)
    S[same_family] = 0.70
    same_port = (
        np.isfinite(dport[i])
        & np.isfinite(dport[j])
        & (dport[i] == dport[j])
    )
    S[same_port] = 1.00

    P = (proto[i] == proto[j]).astype(np.float32)

    cosine = np.einsum("ij,ij->i", Z[i], Z[j])
    cosine = np.clip(cosine, -1.0, 1.0)
    B = ((cosine + 1.0) / 2.0).astype(np.float32)

    C = np.ones(len(pairs), dtype=np.float32)
    return C, S, P, B


def scalar_bdg_edge_attr(C, S, P, B):
    return ((C + S + P + B) / 4.0).astype(np.float32).reshape(-1, 1)


def vector_bdg_edge_attr(C, S, P, B):
    if len(C) == 0:
        return np.empty((0, 4), dtype=np.float32)
    return np.column_stack([C, S, P, B]).astype(np.float32)


def scalar_cg_edge_attr(n_edges):
    return np.ones((n_edges, 1), dtype=np.float32)


def vector_cg_edge_attr(n_edges):
    if n_edges == 0:
        return np.empty((0, 4), dtype=np.float32)
    out = np.zeros((n_edges, 4), dtype=np.float32)
    out[:, 0] = 1.0
    return out


def undirected_edges(pairs, edge_features):
    """Duplicate undirected edge features in both directions."""
    pairs = np.asarray(pairs, dtype=np.int64)
    edge_features = np.asarray(edge_features, dtype=np.float32)

    if edge_features.ndim == 1:
        edge_features = edge_features.reshape(-1, 1)

    edge_dim = edge_features.shape[1] if edge_features.ndim == 2 else 1

    if len(pairs) == 0:
        return (
            torch.empty((2, 0), dtype=torch.long),
            torch.empty((0, edge_dim), dtype=torch.float32),
        )

    if len(edge_features) != len(pairs):
        raise ValueError("edge_features and pairs must have the same length")

    src = np.concatenate([pairs[:, 0], pairs[:, 1]])
    dst = np.concatenate([pairs[:, 1], pairs[:, 0]])
    ef = np.concatenate([edge_features, edge_features], axis=0)

    return (
        torch.tensor(np.vstack([src, dst]), dtype=torch.long),
        torch.tensor(ef, dtype=torch.float32),
    )


In [ ]:

# CELL 9 — PREPARE WINDOW EXPERIMENT + IDENTICAL-TOPOLOGY CACHE

def prepare_window_experiment(base_df, window_seconds, seed=42):
    windowed = add_window_ids(base_df, window_seconds, MAX_FLOWS_PER_GRAPH)
    windowed = safe_graph_split(windowed, seed)
    counts = windowed.groupby("_graph_id")["_split"].nunique()
    if (counts > 1).any():
        raise RuntimeError("Split leakage detected.")

    train_df = windowed[windowed["_split"] == "train"].copy()
    val_df = windowed[windowed["_split"] == "val"].copy()
    test_df = windowed[windowed["_split"] == "test"].copy()

    flow_pp = FlowPreprocessor().fit(train_df)
    bdd_pp = BDDTrafficPreprocessor().fit(train_df)

    return {
        "windowed": windowed,
        "train_df": train_df,
        "val_df": val_df,
        "test_df": test_df,
        "flow_pp": flow_pp,
        "bdd_pp": bdd_pp,
    }


def prepare_split_cache(split_df, flow_pp, bdd_pp):
    cache = {}
    for gid, g in split_df.groupby("_graph_id", sort=False):
        g = g.sort_values(["timestamp", "_row_id"]).reset_index(drop=True)
        X = flow_pp.transform_scaled(g)
        Z = bdd_pp.transform(g)
        pair_set = endpoint_candidate_pairs(g, CG_NEIGHBORS_PER_ENDPOINT)
        pairs = (
            np.asarray(sorted(pair_set), dtype=np.int64)
            if pair_set
            else np.empty((0, 2), dtype=np.int64)
        )
        C, S, P, B = compute_bdg_components(g, Z, pairs)
        cache[gid] = {
            "gdf": g,
            "X": X,
            "Z": Z,
            "pairs": pairs,
            "C": C,
            "S": S,
            "P": P,
            "B": B,
        }
    return cache


def _graph_stats(d, graph_type):
    deg = np.zeros(d.n_nodes, dtype=np.int64)
    if d.edge_index.numel() > 0:
        np.add.at(deg, d.edge_index[0].numpy(), 1)
    avg_degree = (2.0 * d.n_edges_undirected / d.n_nodes) if d.n_nodes else 0.0
    possible = d.n_nodes * (d.n_nodes - 1) / 2
    density = (d.n_edges_undirected / possible) if possible > 0 else 0.0
    isolated_pct = float((deg == 0).mean() * 100) if len(deg) else 0.0
    return {
        "graph_id": d.graph_id,
        "graph_type": graph_type,
        "nodes": d.n_nodes,
        "edges_undirected": d.n_edges_undirected,
        "average_degree": avg_degree,
        "density": density,
        "isolated_node_pct": isolated_pct,
    }


def communication_graphs_from_cache(cache, representation="scalar"):
    graphs, stats = [], []
    for gid, item in cache.items():
        g, X, pairs = item["gdf"], item["X"], item["pairs"]

        if representation == "scalar":
            features = scalar_cg_edge_attr(len(pairs))
        elif representation == "vector":
            features = vector_cg_edge_attr(len(pairs))
        else:
            raise ValueError(f"Unknown representation: {representation}")

        edge_index, edge_attr = undirected_edges(pairs, features)
        d = Data(
            x=torch.tensor(X, dtype=torch.float32),
            edge_index=edge_index,
            edge_attr=edge_attr,
            y=torch.tensor(g["y"].values, dtype=torch.long),
        )
        d.graph_id = str(gid)
        d.n_nodes = len(g)
        d.n_edges_undirected = len(pairs)
        graphs.append(d)
        stats.append(_graph_stats(d, "communication"))
    return graphs, pd.DataFrame(stats)


def bdg_graphs_from_cache(cache, representation="scalar"):
    graphs, stats = [], []
    for gid, item in cache.items():
        g, X, pairs = item["gdf"], item["X"], item["pairs"]
        C, S, P, B = item["C"], item["S"], item["P"], item["B"]

        if representation == "scalar":
            features = scalar_bdg_edge_attr(C, S, P, B)
        elif representation == "vector":
            features = vector_bdg_edge_attr(C, S, P, B)
        else:
            raise ValueError(f"Unknown representation: {representation}")

        edge_index, edge_attr = undirected_edges(pairs, features)
        d = Data(
            x=torch.tensor(X, dtype=torch.float32),
            edge_index=edge_index,
            edge_attr=edge_attr,
            y=torch.tensor(g["y"].values, dtype=torch.long),
        )
        d.graph_id = str(gid)
        d.n_nodes = len(g)
        d.n_edges_undirected = len(pairs)
        graphs.append(d)
        stats.append(_graph_stats(d, "bdg"))
    return graphs, pd.DataFrame(stats)





def assert_identical_topology(cg_graphs, bdg_graphs, label):
    assert len(cg_graphs) == len(bdg_graphs), f"{label}: graph-count mismatch"
    for cg, bdg in zip(cg_graphs, bdg_graphs):
        assert cg.num_nodes == bdg.num_nodes, f"{label}: node-count mismatch"
        assert cg.n_edges_undirected == bdg.n_edges_undirected, f"{label}: edge-count mismatch"
        assert torch.equal(cg.edge_index, bdg.edge_index), f"{label}: edge_index mismatch"
    print(f"PASS: {label} CG and BDG have identical topology.")

In [ ]:
# CELL 10 — EDGE-AWARE GAT + METRICS

class CurrentGAT(nn.Module):
    def __init__(self, in_dim, n_classes, edge_dim=1):
        super().__init__()

        self.gat1 = GATConv(
            in_channels=in_dim,
            out_channels=GAT_HIDDEN_CHANNELS,
            heads=GAT_HEADS,
            concat=True,
            dropout=GAT_ATTENTION_DROPOUT,
            edge_dim=edge_dim,
            add_self_loops=True,
            fill_value=1.0,
        )

        first_dim = GAT_HIDDEN_CHANNELS * GAT_HEADS

        self.gat2 = GATConv(
            in_channels=first_dim,
            out_channels=GAT_HIDDEN_CHANNELS,
            heads=GAT_SECOND_HEADS,
            concat=False,
            dropout=GAT_ATTENTION_DROPOUT,
            edge_dim=edge_dim,
            add_self_loops=True,
            fill_value=1.0,
        )

        self.norm1 = nn.LayerNorm(first_dim)
        self.norm2 = nn.LayerNorm(GAT_HIDDEN_CHANNELS)
        self.fc = nn.Linear(GAT_HIDDEN_CHANNELS, n_classes)

    def forward(self, x, edge_index, edge_attr):
        x = self.gat1(x, edge_index, edge_attr=edge_attr)
        x = self.norm1(x)
        x = F.elu(x)
        x = F.dropout(x, p=GAT_DROPOUT, training=self.training)

        x = self.gat2(x, edge_index, edge_attr=edge_attr)
        x = self.norm2(x)
        x = F.elu(x)
        x = F.dropout(x, p=GAT_DROPOUT, training=self.training)

        return self.fc(x)



class ResidualGAT(nn.Module):
    def __init__(self, in_dim, n_classes, edge_dim=1):
        super().__init__()

        self.input_proj = nn.Linear(in_dim, GAT_HIDDEN_CHANNELS)

        self.gat1 = GATConv(
            in_channels=in_dim,
            out_channels=GAT_HIDDEN_CHANNELS,
            heads=GAT_HEADS,
            concat=True,
            dropout=GAT_ATTENTION_DROPOUT,
            edge_dim=edge_dim,
            add_self_loops=True,
            fill_value=1.0,
        )

        first_dim = GAT_HIDDEN_CHANNELS * GAT_HEADS

        self.gat1_reduce = nn.Linear(
            first_dim,
            GAT_HIDDEN_CHANNELS,
        )

        self.gat2 = GATConv(
            in_channels=GAT_HIDDEN_CHANNELS,
            out_channels=GAT_HIDDEN_CHANNELS,
            heads=GAT_SECOND_HEADS,
            concat=False,
            dropout=GAT_ATTENTION_DROPOUT,
            edge_dim=edge_dim,
            add_self_loops=True,
            fill_value=1.0,
        )

        self.norm1 = nn.LayerNorm(GAT_HIDDEN_CHANNELS)
        self.norm2 = nn.LayerNorm(GAT_HIDDEN_CHANNELS)

        self.classifier = nn.Sequential(
            nn.Linear(GAT_HIDDEN_CHANNELS, ARCH_MLP_HIDDEN),
            nn.ReLU(),
            nn.Dropout(ARCH_MLP_DROPOUT),
            nn.Linear(ARCH_MLP_HIDDEN, n_classes),
        )

    def forward(self, x, edge_index, edge_attr):
        residual = self.input_proj(x)

        h1 = self.gat1(
            x,
            edge_index,
            edge_attr=edge_attr,
        )
        h1 = self.gat1_reduce(h1)
        h1 = self.norm1(h1 + residual)
        h1 = F.elu(h1)
        h1 = F.dropout(
            h1,
            p=GAT_DROPOUT,
            training=self.training,
        )

        h2 = self.gat2(
            h1,
            edge_index,
            edge_attr=edge_attr,
        )
        h2 = self.norm2(h2 + h1)
        h2 = F.elu(h2)
        h2 = F.dropout(
            h2,
            p=GAT_DROPOUT,
            training=self.training,
        )

        return self.classifier(h2)


class ShallowGAT(nn.Module):
    def __init__(self, in_dim, n_classes, edge_dim=1):
        super().__init__()

        self.gat1 = GATConv(
            in_channels=in_dim,
            out_channels=GAT_HIDDEN_CHANNELS,
            heads=GAT_HEADS,
            concat=True,
            dropout=GAT_ATTENTION_DROPOUT,
            edge_dim=edge_dim,
            add_self_loops=True,
            fill_value=1.0,
        )

        first_dim = GAT_HIDDEN_CHANNELS * GAT_HEADS

        self.norm1 = nn.LayerNorm(first_dim)

        self.classifier = nn.Sequential(
            nn.Linear(first_dim, ARCH_MLP_HIDDEN),
            nn.ReLU(),
            nn.Dropout(ARCH_MLP_DROPOUT),
            nn.Linear(ARCH_MLP_HIDDEN, n_classes),
        )

    def forward(self, x, edge_index, edge_attr):
        x = self.gat1(
            x,
            edge_index,
            edge_attr=edge_attr,
        )
        x = self.norm1(x)
        x = F.elu(x)
        x = F.dropout(
            x,
            p=GAT_DROPOUT,
            training=self.training,
        )
        return self.classifier(x)


def build_gat_model(architecture_name, in_dim, n_classes, edge_dim=1):
    if architecture_name == "current_2gat_linear":
        return CurrentGAT(in_dim=in_dim, n_classes=n_classes, edge_dim=edge_dim)

    if architecture_name == "residual_2gat_mlp":
        return ResidualGAT(in_dim=in_dim, n_classes=n_classes, edge_dim=edge_dim)

    if architecture_name == "shallow_1gat_mlp":
        return ShallowGAT(in_dim=in_dim, n_classes=n_classes, edge_dim=edge_dim)

    raise ValueError(f"Unknown architecture: {architecture_name}")


# Backward-compatible alias for existing cells.
EdgeAwareGAT = CurrentGAT


def metric_bundle(y_true, y_pred):
    pm, rm, fm, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    pw, rw, fw, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": pm,
        "recall_macro": rm,
        "f1_macro": fm,
        "precision_weighted": pw,
        "recall_weighted": rw,
        "f1_weighted": fw,
    }


@torch.no_grad()
def predict_model(model, loader):
    model.eval()
    ys, ps = [], []

    for batch in loader:
        batch = batch.to(DEVICE)
        logits = model(batch.x, batch.edge_index, batch.edge_attr)
        pred = logits.argmax(1)
        ys.append(batch.y.cpu().numpy())
        ps.append(pred.cpu().numpy())

    return np.concatenate(ys), np.concatenate(ps)


def train_gat(train_graphs, val_graphs, epochs, seed=42, architecture_name="current_2gat_linear",):
    seed_everything(seed)

    model = build_gat_model(
        architecture_name=architecture_name,
        in_dim=train_graphs[0].num_node_features,
        n_classes=NUM_CLASSES,
        edge_dim=train_graphs[0].edge_attr.shape[1],
    ).to(DEVICE)

    train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_graphs, batch_size=BATCH_SIZE, shuffle=False)

    if USE_CLASS_WEIGHTS:
        ytrain = np.concatenate([g.y.detach().cpu().numpy() for g in train_graphs])
        present = np.unique(ytrain)
        weights = np.ones(NUM_CLASSES, dtype=np.float32)
        calculated = compute_class_weight(
            class_weight="balanced", classes=present, y=ytrain
        )
        for cls, w in zip(present, calculated):
            weights[int(cls)] = float(w)
        weights = torch.tensor(weights, dtype=torch.float32, device=DEVICE)
    else:
        weights = None

    criterion = nn.CrossEntropyLoss(weight=weights)
    opt = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    best_state = None
    best_macro_f1 = -1.0
    stale = 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, total_nodes = 0.0, 0

        for batch in train_loader:
            batch = batch.to(DEVICE)
            opt.zero_grad()

            logits = model(batch.x, batch.edge_index, batch.edge_attr)
            loss = criterion(logits, batch.y)
            loss.backward()
            opt.step()

            total_loss += loss.item() * batch.y.numel()
            total_nodes += batch.y.numel()

        yv, pv = predict_model(model, val_loader)
        vm = metric_bundle(yv, pv)

        history.append({
            "epoch": epoch,
            "train_loss": total_loss / max(total_nodes, 1),
            **{f"val_{k}": v for k, v in vm.items()},
        })

        if vm["f1_macro"] > best_macro_f1:
            best_macro_f1 = vm["f1_macro"]
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            stale = 0
        else:
            stale += 1

        if stale >= PATIENCE:
            break

    if best_state is None:
        raise RuntimeError("No valid GAT checkpoint was produced.")

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)



# Temporal-window ablation only

The only ablation is the graph-window duration: 10, 30, 60, 300, and 600 seconds.

For every window, CG and BDG are built from the same cache and their topology is checked for exact equality. Both use the same GAT. A single common window is selected using the mean of CG and BDG validation Macro-F1.


In [ ]:

# CELL 11 — TEMPORAL-WINDOW ABLATION ONLY

ablation_rows = []

for ws in WINDOW_SECONDS:
    print("\n" + "=" * 90)
    print("WINDOW:", ws, "seconds")
    print("=" * 90)
    t0 = time.time()

    exp = prepare_window_experiment(data, ws, SEED)
    train_cache = prepare_split_cache(exp["train_df"], exp["flow_pp"], exp["bdd_pp"])
    val_cache = prepare_split_cache(exp["val_df"], exp["flow_pp"], exp["bdd_pp"])

    cg_train, cg_train_stats = communication_graphs_from_cache(train_cache)
    bdg_train, bdg_train_stats = bdg_graphs_from_cache(train_cache)
    cg_val, cg_val_stats = communication_graphs_from_cache(val_cache)
    bdg_val, bdg_val_stats = bdg_graphs_from_cache(val_cache)

    assert_identical_topology(cg_train, bdg_train, f"{ws}s train")
    assert_identical_topology(cg_val, bdg_val, f"{ws}s validation")

    print("Training Communication Graph + GAT...")
    cg_model, _ = train_gat(cg_train, cg_val, ABLATION_EPOCHS, SEED)
    cg_loader = DataLoader(cg_val, batch_size=BATCH_SIZE, shuffle=False)
    cg_y, cg_p = predict_model(cg_model, cg_loader)
    cg_metrics = metric_bundle(cg_y, cg_p)

    print("Training Behavioral Dependency Graph + GAT...")
    bdg_model, _ = train_gat(bdg_train, bdg_val, ABLATION_EPOCHS, SEED)
    bdg_loader = DataLoader(bdg_val, batch_size=BATCH_SIZE, shuffle=False)
    bdg_y, bdg_p = predict_model(bdg_model, bdg_loader)
    bdg_metrics = metric_bundle(bdg_y, bdg_p)

    common = (cg_metrics["f1_macro"] + bdg_metrics["f1_macro"]) / 2.0
    cg_stats = pd.concat([cg_train_stats, cg_val_stats], ignore_index=True)
    bdg_stats = pd.concat([bdg_train_stats, bdg_val_stats], ignore_index=True)

    for col in ["edges_undirected", "average_degree", "density", "isolated_node_pct"]:
        assert np.isclose(cg_stats[col].mean(), bdg_stats[col].mean()), f"{ws}s topology-stat mismatch: {col}"

    ablation_rows.append({
        "window_seconds": ws,
        "cg_validation_accuracy": cg_metrics["accuracy"],
        "cg_validation_f1_macro": cg_metrics["f1_macro"],
        "cg_validation_f1_weighted": cg_metrics["f1_weighted"],
        "bdg_validation_accuracy": bdg_metrics["accuracy"],
        "bdg_validation_f1_macro": bdg_metrics["f1_macro"],
        "bdg_validation_f1_weighted": bdg_metrics["f1_weighted"],
        "common_validation_macro_f1": common,
        "mean_edges": cg_stats["edges_undirected"].mean(),
        "mean_degree": cg_stats["average_degree"].mean(),
        "mean_density": cg_stats["density"].mean(),
        "mean_isolated_pct": cg_stats["isolated_node_pct"].mean(),
        "runtime_seconds": time.time() - t0,
    })

    del cg_model, bdg_model, cg_train, bdg_train, cg_val, bdg_val, train_cache, val_cache, exp
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

ablation_df = pd.DataFrame(ablation_rows).sort_values("common_validation_macro_f1", ascending=False)
display(ablation_df)
ablation_df.to_csv(RESULT_DIR / "04_temporal_window_ablation.csv", index=False)

BEST_WINDOW_SECONDS = int(ablation_df.iloc[0]["window_seconds"])
print("\nLOCKED COMMON WINDOW:", BEST_WINDOW_SECONDS, "seconds")

In [ ]:
# CELL 12 — PLOT WINDOW ABLATION
p = ablation_df.sort_values("window_seconds")

plt.figure(figsize=(9,5))
plt.plot(p["window_seconds"], p["cg_validation_f1_macro"], marker="o", label="CG Macro-F1")
plt.plot(p["window_seconds"], p["bdg_validation_f1_macro"], marker="o", label="BDG Macro-F1")
plt.plot(p["window_seconds"], p["common_validation_macro_f1"], marker="o", linestyle="--", label="Common selection score")
plt.xlabel("Temporal window (seconds)")
plt.ylabel("Validation Macro-F1")
plt.title("Temporal Window Ablation: CG vs BDG")
plt.legend()
plt.grid(alpha=.25)
plt.tight_layout()
plt.savefig(RESULT_DIR / "window_ablation_cg_vs_bdg.png", dpi=300)
plt.show()

# Final locked comparison

After validation, one common temporal window is locked. A second validation-only
ablation compares two CC-BDG edge representations while keeping topology fixed:

- **Scalar CSPB:** `(C + S + P + B) / 4`;
- **Vector CSPB:** `[C, S, P, B]`.

The representation with the highest mean validation Macro-F1 is locked before
the final 10-seed test. The test set is not used for representation selection.

For a fair final comparison, CG uses the same edge dimensionality as the selected
CC-BDG representation:

- scalar mode: CG edge = `[1]`;
- vector mode: CG edge = `[1,0,0,0]`.

Thus CG and CC-BDG retain identical nodes and topology; only edge semantics differ.


In [ ]:

# CELL 13 — LOCK FINAL SPLIT + BUILD IDENTICAL-TOPOLOGY CG AND BDG
final_exp = prepare_window_experiment(data, BEST_WINDOW_SECONDS, SEED)

print("Train flows:", len(final_exp["train_df"]))
print("Val flows:", len(final_exp["val_df"]))
print("Test flows:", len(final_exp["test_df"]))
print("Locked common window:", BEST_WINDOW_SECONDS)

print("\nPreprocessing:")
print(json.dumps(final_exp["flow_pp"].report(), indent=2))
with open(RESULT_DIR / "preprocessing_report.json", "w") as f:
    json.dump(final_exp["flow_pp"].report(), f, indent=2)

split_dist = final_exp["windowed"].groupby(["_split", TARGET_COLUMN]).size().reset_index(name="count")
display(split_dist)
split_dist.to_csv(RESULT_DIR / "06_split_class_distribution.csv", index=False)

communication_sets = {}
bdg_sets = {}
final_caches = {}

for split in ["train", "val", "test"]:
    print(f"Preparing final {split} cache...")
    cache = prepare_split_cache(final_exp[f"{split}_df"], final_exp["flow_pp"], final_exp["bdd_pp"])
    cg_graphs, cg_stats = communication_graphs_from_cache(cache, representation="scalar")
    bdg_graphs, bdg_stats = bdg_graphs_from_cache(cache, representation="scalar")
    assert_identical_topology(cg_graphs, bdg_graphs, split)

    communication_sets[split] = cg_graphs
    communication_sets[f"{split}_stats"] = cg_stats
    bdg_sets[split] = bdg_graphs
    bdg_sets[f"{split}_stats"] = bdg_stats
    final_caches[split] = cache
    gc.collect()

graph_stats = pd.concat([
    communication_sets["train_stats"].assign(split="train"),
    communication_sets["val_stats"].assign(split="val"),
    communication_sets["test_stats"].assign(split="test"),
    bdg_sets["train_stats"].assign(split="train"),
    bdg_sets["val_stats"].assign(split="val"),
    bdg_sets["test_stats"].assign(split="test"),
], ignore_index=True)

summary_stats = graph_stats.groupby(["graph_type", "split"]).agg(
    graphs=("graph_id", "count"),
    mean_nodes=("nodes", "mean"),
    median_nodes=("nodes", "median"),
    mean_edges=("edges_undirected", "mean"),
    median_edges=("edges_undirected", "median"),
    mean_degree=("average_degree", "mean"),
    mean_density=("density", "mean"),
    mean_isolated_pct=("isolated_node_pct", "mean"),
).reset_index()

display(summary_stats)

for split in ["train", "val", "test"]:
    cg = summary_stats[(summary_stats["graph_type"] == "communication") & (summary_stats["split"] == split)].iloc[0]
    bdg = summary_stats[(summary_stats["graph_type"] == "bdg") & (summary_stats["split"] == split)].iloc[0]
    for col in ["mean_nodes", "mean_edges", "mean_degree", "mean_density", "mean_isolated_pct"]:
        assert np.isclose(cg[col], bdg[col]), f"{split}: topology mismatch in {col}"

print("PASS: CG and BDG final topology statistics are identical.")

graph_stats.to_csv(RESULT_DIR / "07_graph_statistics.csv", index=False)
summary_stats.to_csv(RESULT_DIR / "08_graph_statistics_summary.csv", index=False)

# Edge-attribute audit
weight_rows = []
for split in ["train", "val", "test"]:
    vals = [g.edge_attr.cpu().numpy().reshape(-1) for g in bdg_sets[split] if g.edge_attr.numel() > 0]
    vals = np.concatenate(vals) if vals else np.array([], dtype=np.float32)
    if len(vals):
        weight_rows.append({
            "split": split,
            "min_edge_attr": float(vals.min()),
            "mean_edge_attr": float(vals.mean()),
            "median_edge_attr": float(np.median(vals)),
            "max_edge_attr": float(vals.max()),
            "std_edge_attr": float(vals.std()),
        })

bdg_edge_attr_stats = pd.DataFrame(weight_rows)
display(bdg_edge_attr_stats)
bdg_edge_attr_stats.to_csv(RESULT_DIR / "09_bdg_edge_attr_statistics.csv", index=False)

for split in ["train", "val", "test"]:
    for g in communication_sets[split]:
        if g.edge_attr.numel() > 0:
            assert torch.all(g.edge_attr == 1.0)
print("PASS: every CG edge attribute is exactly 1.0.")

In [ ]:
# SIX GRAPH COLLECTIONS CREATED AFTER THE LOCKED SPLIT
six_collections = pd.DataFrame([
    {"split": "train", "representation": "Communication Graph", "graphs": len(communication_sets["train"])},
    {"split": "validation", "representation": "Communication Graph", "graphs": len(communication_sets["val"])},
    {"split": "test", "representation": "Communication Graph", "graphs": len(communication_sets["test"])},
    {"split": "train", "representation": "Behavioral Dependency Graph", "graphs": len(bdg_sets["train"])},
    {"split": "validation", "representation": "Behavioral Dependency Graph", "graphs": len(bdg_sets["val"])},
    {"split": "test", "representation": "Behavioral Dependency Graph", "graphs": len(bdg_sets["test"])},
])

display(six_collections)
six_collections.to_csv(RESULT_DIR / "09_six_graph_collections.csv", index=False)

## Split isolation check

The split-isolation audit is executed inside `prepare_window_experiment()` and again immediately before the final graph collections are constructed. No graph edge can cross Train, Validation, and Test boundaries.

In [ ]:
# FINAL METHOD CONFIGURATION

final_method_config = {
    "common_window_seconds": BEST_WINDOW_SECONDS,
    "cg_edge_definition": "same shared-endpoint topology as CC-BDG",
    "bdg_edge_definition": "same shared-endpoint edges as CG",
    "bdg_behavioral_components": [
        "communication",
        "service",
        "protocol",
        "traffic_pattern",
    ],
    "bdg_representation_candidates": [
        "scalar: (C+S+P+B)/4",
        "vector: [C,S,P,B]",
    ],
    "representation_selection": "validation Macro-F1 only",
    "test_used_for_representation_selection": False,
    "bdg_threshold_selection": False,
    "bdg_component_selection": False,
    "edge_filtering": False,
    "topology_invariant": "CG and BDG have identical nodes and edges.",
}

print(json.dumps(final_method_config, indent=2))

with open(RESULT_DIR / "final_method_config.json", "w") as f:
    json.dump(final_method_config, f, indent=2)


## Validation-only CC-BDG representation ablation

This ablation compares only the **edge representation**. The split, temporal
window, node features, shared-endpoint topology, optimizer, loss, and all other
training settings are unchanged.

Two predefined representations are evaluated on the validation set:

1. **Scalar CSPB:** `(C + S + P + B) / 4`
2. **Vector CSPB:** `[C, S, P, B]`

The comparison uses the fixed `current_2gat_linear` architecture and seeds
42, 52, and 62. The representation with the highest mean validation Macro-F1
is locked. Only after that lock is the GAT architecture ablation performed.
The test set is never used for representation selection.


In [ ]:
# VALIDATION-ONLY SCALAR VS VECTOR CSPB ABLATION
REPRESENTATION_CANDIDATES = ["scalar", "vector"]
REPRESENTATION_ABLATION_SEEDS = [42, 52, 62]
REPRESENTATION_ABLATION_ARCH = "current_2gat_linear"

representation_rows = []

for representation in REPRESENTATION_CANDIDATES:
    print("\n" + "=" * 90)
    print("CC-BDG EDGE REPRESENTATION:", representation)
    print("=" * 90)

    rep_train, _ = bdg_graphs_from_cache(
        final_caches["train"],
        representation=representation,
    )
    rep_val, _ = bdg_graphs_from_cache(
        final_caches["val"],
        representation=representation,
    )

    for seed in REPRESENTATION_ABLATION_SEEDS:
        print("Validation representation ablation:", representation, "seed", seed)

        model, history = train_gat(
            rep_train,
            rep_val,
            ABLATION_EPOCHS,
            seed,
            architecture_name=REPRESENTATION_ABLATION_ARCH,
        )

        val_loader = DataLoader(
            rep_val,
            batch_size=BATCH_SIZE,
            shuffle=False,
        )
        y_val, p_val = predict_model(model, val_loader)
        metrics = metric_bundle(y_val, p_val)

        representation_rows.append({
            "representation": representation,
            "seed": seed,
            "edge_dim": int(rep_train[0].edge_attr.shape[1]),
            "validation_accuracy": metrics["accuracy"],
            "validation_precision_macro": metrics["precision_macro"],
            "validation_recall_macro": metrics["recall_macro"],
            "validation_f1_macro": metrics["f1_macro"],
            "validation_f1_weighted": metrics["f1_weighted"],
        })

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    del rep_train, rep_val
    gc.collect()

representation_ablation_seed_df = pd.DataFrame(representation_rows)
representation_ablation_summary = (
    representation_ablation_seed_df
    .groupby(["representation", "edge_dim"], as_index=False)
    .agg(
        validation_accuracy_mean=("validation_accuracy", "mean"),
        validation_accuracy_std=("validation_accuracy", "std"),
        validation_precision_macro_mean=("validation_precision_macro", "mean"),
        validation_precision_macro_std=("validation_precision_macro", "std"),
        validation_recall_macro_mean=("validation_recall_macro", "mean"),
        validation_recall_macro_std=("validation_recall_macro", "std"),
        validation_f1_macro_mean=("validation_f1_macro", "mean"),
        validation_f1_macro_std=("validation_f1_macro", "std"),
        validation_f1_weighted_mean=("validation_f1_weighted", "mean"),
        validation_f1_weighted_std=("validation_f1_weighted", "std"),
    )
    .sort_values("validation_f1_macro_mean", ascending=False)
    .reset_index(drop=True)
)

print("\nSCALAR VS VECTOR — VALIDATION MEAN ± STD")
display(representation_ablation_summary)

representation_ablation_seed_df.to_csv(
    RESULT_DIR / "10_cspb_representation_ablation_seed_results.csv",
    index=False,
)
representation_ablation_summary.to_csv(
    RESULT_DIR / "11_cspb_representation_ablation_summary.csv",
    index=False,
)

BEST_BDG_REPRESENTATION = representation_ablation_summary.iloc[0]["representation"]
BEST_EDGE_DIM = int(representation_ablation_summary.iloc[0]["edge_dim"])

print("\nLOCKED CC-BDG REPRESENTATION:", BEST_BDG_REPRESENTATION)
print("LOCKED EDGE DIMENSION:", BEST_EDGE_DIM)
print(
    "Mean validation Macro-F1:",
    float(representation_ablation_summary.iloc[0]["validation_f1_macro_mean"]),
)

# Rebuild BOTH graph methods with the locked dimensionality.
# Topology remains identical.
communication_sets = {}
bdg_sets = {}

for split in ["train", "val", "test"]:
    cg_graphs, cg_stats = communication_graphs_from_cache(
        final_caches[split],
        representation=BEST_BDG_REPRESENTATION,
    )
    bdg_graphs, bdg_stats = bdg_graphs_from_cache(
        final_caches[split],
        representation=BEST_BDG_REPRESENTATION,
    )

    assert_identical_topology(cg_graphs, bdg_graphs, f"locked {split}")
    assert cg_graphs[0].edge_attr.shape[1] == BEST_EDGE_DIM
    assert bdg_graphs[0].edge_attr.shape[1] == BEST_EDGE_DIM

    communication_sets[split] = cg_graphs
    communication_sets[f"{split}_stats"] = cg_stats
    bdg_sets[split] = bdg_graphs
    bdg_sets[f"{split}_stats"] = bdg_stats

# Verify CG edge semantics after representation lock.
for split in ["train", "val", "test"]:
    for g in communication_sets[split]:
        if g.edge_attr.numel() == 0:
            continue
        if BEST_BDG_REPRESENTATION == "scalar":
            assert torch.all(g.edge_attr == 1.0)
        else:
            assert torch.all(g.edge_attr[:, 0] == 1.0)
            assert torch.all(g.edge_attr[:, 1:] == 0.0)

final_method_config["locked_bdg_representation"] = BEST_BDG_REPRESENTATION
final_method_config["locked_edge_dim"] = BEST_EDGE_DIM
with open(RESULT_DIR / "final_method_config.json", "w") as f:
    json.dump(final_method_config, f, indent=2)

print("PASS: representation locked using validation only; final graph sets rebuilt.")


# GAT architecture ablation — validation only

The CC-BDG edge representation has already been locked using validation data only.

Three predefined GAT variants are now compared on the locked CC-BDG validation set:

1. `current_2gat_linear` — existing two-layer GAT + linear classifier.
2. `residual_2gat_mlp` — two-layer GAT + residual connections + MLP classifier.
3. `shallow_1gat_mlp` — one GAT layer + MLP classifier.

The architecture with the highest validation Macro-F1 is locked. The same locked
architecture is then applied to both CG and CC-BDG in the final 10-seed test.


In [ ]:

# GAT ARCHITECTURE ABLATION — CC-BDG VALIDATION ONLY

architecture_rows = []

for architecture_name in ARCHITECTURE_CANDIDATES:

    print("\n" + "=" * 90)
    print("ARCHITECTURE:", architecture_name)
    print("=" * 90)

    model, history = train_gat(
        bdg_sets["train"],
        bdg_sets["val"],
        ABLATION_EPOCHS,
        ARCH_ABLATION_SEED,
        architecture_name=architecture_name,
    )

    val_loader = DataLoader(
        bdg_sets["val"],
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    y_val, p_val = predict_model(
        model,
        val_loader,
    )

    metrics = metric_bundle(
        y_val,
        p_val,
    )

    n_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    architecture_rows.append({
        "architecture": architecture_name,
        "validation_accuracy": metrics["accuracy"],
        "validation_precision_macro": metrics["precision_macro"],
        "validation_recall_macro": metrics["recall_macro"],
        "validation_f1_macro": metrics["f1_macro"],
        "validation_f1_weighted": metrics["f1_weighted"],
        "trainable_parameters": n_params,
    })

    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


architecture_ablation_df = (
    pd.DataFrame(architecture_rows)
    .sort_values(
        "validation_f1_macro",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(architecture_ablation_df)

architecture_ablation_df.to_csv(
    RESULT_DIR / "gat_architecture_ablation.csv",
    index=False,
)

BEST_GAT_ARCHITECTURE = (
    architecture_ablation_df.iloc[0]["architecture"]
)

print(
    "\nLOCKED GAT ARCHITECTURE:",
    BEST_GAT_ARCHITECTURE,
)


In [ ]:

# CELL 14 — FINAL MULTI-SEED CG-GAT vs BDG-GAT
def evaluate_graph_method(sets, method, seed):
    model, history = train_gat(
        sets["train"],
        sets["val"],
        FINAL_EPOCHS,
        seed,
        architecture_name=BEST_GAT_ARCHITECTURE,
    )

    test_loader = DataLoader(sets["test"], batch_size=BATCH_SIZE, shuffle=False)
    yt, yp = predict_model(model, test_loader)

    metrics = metric_bundle(yt, yp)

    report = pd.DataFrame(
        classification_report(
            yt, yp,
            labels=np.arange(NUM_CLASSES),
            target_names=CLASS_NAMES,
            output_dict=True,
            zero_division=0
        )
    ).T

    history.to_csv(RESULT_DIR / f"history_{method}_seed{seed}.csv", index=False)
    report.to_csv(RESULT_DIR / f"per_class_{method}_seed{seed}.csv")

    return model, history, yt, yp, metrics, report

seed_rows = []
first_seed_details = {}
graph_reports_by_seed = {}

for seed in FINAL_SEEDS:
    for method, sets in [
        ("communication_gat", communication_sets),
        ("bdg_gat", bdg_sets),
    ]:
        print("Running", method, "seed", seed)
        result = evaluate_graph_method(sets, method, seed)
        model, history, yt, yp, metrics, report = result

        seed_rows.append({
            "seed": seed,
            "method": method,
            **metrics
        })

        graph_reports_by_seed[(method, seed)] = report.copy()

        if seed == FINAL_SEEDS[0]:
            first_seed_details[method] = {
                "model": model,
                "history": history,
                "y_true": yt,
                "y_pred": yp,
                "metrics": metrics,
                "report": report,
            }
        else:
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

seed_results = pd.DataFrame(seed_rows)
display(seed_results)
seed_results.to_csv(RESULT_DIR / "07_final_gat_seed_results.csv", index=False)

summary = seed_results.groupby("method").agg(
    accuracy_mean=("accuracy","mean"),
    accuracy_std=("accuracy","std"),
    precision_macro_mean=("precision_macro","mean"),
    precision_macro_std=("precision_macro","std"),
    recall_macro_mean=("recall_macro","mean"),
    recall_macro_std=("recall_macro","std"),
    f1_macro_mean=("f1_macro","mean"),
    f1_macro_std=("f1_macro","std"),
    precision_weighted_mean=("precision_weighted","mean"),
    precision_weighted_std=("precision_weighted","std"),
    recall_weighted_mean=("recall_weighted","mean"),
    recall_weighted_std=("recall_weighted","std"),
    f1_weighted_mean=("f1_weighted","mean"),
    f1_weighted_std=("f1_weighted","std"),
).reset_index()

display(summary)
summary.to_csv(RESULT_DIR / "08_final_gat_mean_std_summary.csv", index=False)

# Flow-level LightGBM baseline

The graph ablation is already complete at this stage and the common temporal window has been locked.

LightGBM therefore **does not participate in graph-window selection**. It uses the exact flow rows contained in the locked graph-isolated train, validation, and test partitions.

## Feature fairness

LightGBM uses the same final numerical feature columns selected by `final_exp["flow_pp"]`, with the same train-fitted imputer. StandardScaler is not applied because tree-based models do not require feature scaling.

LightGBM receives no graph topology, shared-endpoint relationships, service dependency, protocol dependency, or behavioral edge attributes.

The purpose is to compare the graph methods against a strong **flow-only** baseline while preserving the same dataset partition and evaluation metrics.


In [ ]:
# CELL 15 — PREPARE LOCKED FLOW-LEVEL LIGHTGBM DATA

def prepare_flow_baseline_arrays(final_exp):
    """
    Use the exact rows from the already locked graph-isolated split.
    No re-splitting occurs here.
    """
    pp = final_exp["flow_pp"]

    train_df = final_exp["train_df"].copy()
    val_df = final_exp["val_df"].copy()
    test_df = final_exp["test_df"].copy()

    # LightGBM uses the same columns and train-fitted imputer but no scaling.
    X_train_lgbm = pp.transform_imputed(train_df)
    X_val_lgbm = pp.transform_imputed(val_df)
    X_test_lgbm = pp.transform_imputed(test_df)

    y_train = train_df["y"].to_numpy(dtype=np.int64)
    y_val = val_df["y"].to_numpy(dtype=np.int64)
    y_test = test_df["y"].to_numpy(dtype=np.int64)

    assert len(X_train_lgbm) == len(y_train) == len(train_df)
    assert len(X_val_lgbm) == len(y_val) == len(val_df)
    assert len(X_test_lgbm) == len(y_test) == len(test_df)

    assert X_train_lgbm.shape[1] == len(pp.feature_cols)
    assert X_val_lgbm.shape[1] == len(pp.feature_cols)
    assert X_test_lgbm.shape[1] == len(pp.feature_cols)

    return {
        "X_train_lgbm": X_train_lgbm,
        "X_val_lgbm": X_val_lgbm,
        "X_test_lgbm": X_test_lgbm,
        "y_train": y_train,
        "y_val": y_val,
        "y_test": y_test,
        "feature_cols": list(pp.feature_cols),
    }


flow_baselines = prepare_flow_baseline_arrays(final_exp)

print("Locked flow-level LightGBM arrays")
print("Train:", flow_baselines["X_train_lgbm"].shape)
print("Val:  ", flow_baselines["X_val_lgbm"].shape)
print("Test: ", flow_baselines["X_test_lgbm"].shape)
print("Features:", len(flow_baselines["feature_cols"]))

with open(RESULT_DIR / "13_flow_baseline_feature_columns.json", "w") as f:
    json.dump(flow_baselines["feature_cols"], f, indent=2)


In [ ]:
# CELL 16 — LIGHTGBM DEFINITION

def train_lightgbm_baseline(
    X_train,
    y_train,
    X_val,
    y_val,
    seed,
):
    """
    Fixed LightGBM configuration.
    Validation is used only for early stopping.
    Test labels are never used in fitting or model selection.
    """
    model = LGBMClassifier(
        objective="multiclass",
        num_class=NUM_CLASSES,
        n_estimators=LIGHTGBM_N_ESTIMATORS,
        learning_rate=LIGHTGBM_LEARNING_RATE,
        num_leaves=LIGHTGBM_NUM_LEAVES,
        subsample=LIGHTGBM_SUBSAMPLE,
        subsample_freq=1,
        colsample_bytree=LIGHTGBM_COLSAMPLE,
        class_weight="balanced" if USE_CLASS_WEIGHTS else None,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1,
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="multi_logloss",
        callbacks=[
            early_stopping(
                LIGHTGBM_EARLY_STOPPING_ROUNDS,
                verbose=False,
            ),
            log_evaluation(period=0),
        ],
    )

    return model


In [ ]:
# CELL 17 — FINAL MULTI-SEED LIGHTGBM EVALUATION

def classification_report_df(y_true, y_pred):
    return pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            labels=np.arange(NUM_CLASSES),
            target_names=CLASS_NAMES,
            output_dict=True,
            zero_division=0,
        )
    ).T


baseline_seed_rows = []
baseline_reports_by_seed = {}
baseline_first_seed_details = {}

for seed in FINAL_SEEDS:

    print("Running lightgbm seed", seed)

    lgbm_model = train_lightgbm_baseline(
        flow_baselines["X_train_lgbm"],
        flow_baselines["y_train"],
        flow_baselines["X_val_lgbm"],
        flow_baselines["y_val"],
        seed,
    )

    p_lgbm = lgbm_model.predict(
        flow_baselines["X_test_lgbm"]
    ).astype(np.int64)

    y_lgbm = flow_baselines["y_test"].copy()

    lgbm_metrics = metric_bundle(
        y_lgbm,
        p_lgbm,
    )

    lgbm_report = classification_report_df(
        y_lgbm,
        p_lgbm,
    )

    baseline_seed_rows.append({
        "seed": seed,
        "method": "lightgbm",
        **lgbm_metrics,
    })

    baseline_reports_by_seed[("lightgbm", seed)] = lgbm_report.copy()

    lgbm_report.to_csv(
        RESULT_DIR / f"per_class_lightgbm_seed{seed}.csv"
    )

    pd.DataFrame({
        "feature": flow_baselines["feature_cols"],
        "importance": lgbm_model.feature_importances_,
    }).sort_values(
        "importance",
        ascending=False,
    ).to_csv(
        RESULT_DIR / f"lightgbm_feature_importance_seed{seed}.csv",
        index=False,
    )

    if seed == FINAL_SEEDS[0]:
        baseline_first_seed_details["lightgbm"] = {
            "model": lgbm_model,
            "y_true": y_lgbm,
            "y_pred": p_lgbm,
            "metrics": lgbm_metrics,
            "report": lgbm_report,
        }
    else:
        del lgbm_model

    gc.collect()


baseline_seed_results = pd.DataFrame(baseline_seed_rows)

print("\nLIGHTGBM PER-SEED RESULTS")
display(baseline_seed_results)

baseline_seed_results.to_csv(
    RESULT_DIR / "14_lightgbm_seed_results.csv",
    index=False,
)


# -------------------------------------------------------------
# Combined three-method results
# -------------------------------------------------------------
all_method_seed_results = pd.concat(
    [
        seed_results.copy(),
        baseline_seed_results.copy(),
    ],
    ignore_index=True,
)

display(
    all_method_seed_results.sort_values(
        ["seed", "method"]
    )
)

all_method_seed_results.to_csv(
    RESULT_DIR / "15_all_three_methods_seed_results.csv",
    index=False,
)


all_method_summary = all_method_seed_results.groupby(
    "method"
).agg(
    accuracy_mean=("accuracy", "mean"),
    accuracy_std=("accuracy", "std"),
    precision_macro_mean=("precision_macro", "mean"),
    precision_macro_std=("precision_macro", "std"),
    recall_macro_mean=("recall_macro", "mean"),
    recall_macro_std=("recall_macro", "std"),
    f1_macro_mean=("f1_macro", "mean"),
    f1_macro_std=("f1_macro", "std"),
    precision_weighted_mean=("precision_weighted", "mean"),
    precision_weighted_std=("precision_weighted", "std"),
    recall_weighted_mean=("recall_weighted", "mean"),
    recall_weighted_std=("recall_weighted", "std"),
    f1_weighted_mean=("f1_weighted", "mean"),
    f1_weighted_std=("f1_weighted", "std"),
).reset_index().sort_values(
    "f1_macro_mean",
    ascending=False,
)

print("\nALL THREE METHODS — MULTI-SEED MEAN ± STD")
display(all_method_summary)

all_method_summary.to_csv(
    RESULT_DIR / "16_all_three_methods_mean_std_summary.csv",
    index=False,
)


In [ ]:

# CELL 18 — PER-CLASS RESULTS FOR ALL THREE METHODS

all_reports_by_seed = {}
all_reports_by_seed.update(graph_reports_by_seed)
all_reports_by_seed.update(baseline_reports_by_seed)

method_order = [
    "communication_gat",
    "bdg_gat",
    "lightgbm",
]

# -------------------------------------------------------------
# A. Per-class values for the first final seed
#    (same convention as the original CG-vs-BDG table)
# -------------------------------------------------------------
first_seed = FINAL_SEEDS[0]

first_seed_rows = []

for cls in CLASS_NAMES:
    row = {"class": cls}

    for method in method_order:
        report = all_reports_by_seed[(method, first_seed)]

        row[f"{method}_precision"] = report.loc[
            cls, "precision"
        ]
        row[f"{method}_recall"] = report.loc[
            cls, "recall"
        ]
        row[f"{method}_f1"] = report.loc[
            cls, "f1-score"
        ]
        row[f"{method}_support"] = report.loc[
            cls, "support"
        ]

    first_seed_rows.append(row)

per_class_all_methods_first_seed = pd.DataFrame(
    first_seed_rows
)

print(
    f"\nPER-CLASS RESULTS — SEED {first_seed}"
)
display(
    per_class_all_methods_first_seed
)

per_class_all_methods_first_seed.to_csv(
    RESULT_DIR
    / f"17_per_class_all_three_methods_seed{first_seed}.csv",
    index=False,
)


# -------------------------------------------------------------
# B. Multi-seed per-class mean ± std for all three methods
# -------------------------------------------------------------
long_rows = []

for method in method_order:
    for seed in FINAL_SEEDS:
        report = all_reports_by_seed[(method, seed)]

        for cls in CLASS_NAMES:
            long_rows.append({
                "method": method,
                "seed": seed,
                "class": cls,
                "precision": report.loc[cls, "precision"],
                "recall": report.loc[cls, "recall"],
                "f1": report.loc[cls, "f1-score"],
                "support": report.loc[cls, "support"],
            })

per_class_long = pd.DataFrame(long_rows)

per_class_multiseed_summary = per_class_long.groupby(
    ["method", "class"]
).agg(
    precision_mean=("precision", "mean"),
    precision_std=("precision", "std"),
    recall_mean=("recall", "mean"),
    recall_std=("recall", "std"),
    f1_mean=("f1", "mean"),
    f1_std=("f1", "std"),
    support=("support", "first"),
).reset_index()

print(
    "\nPER-CLASS MULTI-SEED MEAN ± STD — ALL THREE METHODS"
)
display(
    per_class_multiseed_summary
)

per_class_long.to_csv(
    RESULT_DIR
    / "18_per_class_all_three_methods_all_seeds_long.csv",
    index=False,
)

per_class_multiseed_summary.to_csv(
    RESULT_DIR
    / "19_per_class_all_three_methods_mean_std.csv",
    index=False,
)


In [ ]:

# GRAPH-SPECIFIC PER-CLASS DELTA — CG vs CC-BDG (FIRST FINAL SEED)
communication_report = first_seed_details["communication_gat"]["report"]
bdg_report = first_seed_details["bdg_gat"]["report"]

rows = []
for cls in CLASS_NAMES:
    rows.append({
        "class": cls,
        "communication_precision": communication_report.loc[cls,"precision"],
        "communication_recall": communication_report.loc[cls,"recall"],
        "communication_f1": communication_report.loc[cls,"f1-score"],
        "communication_support": communication_report.loc[cls,"support"],
        "bdg_precision": bdg_report.loc[cls,"precision"],
        "bdg_recall": bdg_report.loc[cls,"recall"],
        "bdg_f1": bdg_report.loc[cls,"f1-score"],
        "bdg_support": bdg_report.loc[cls,"support"],
        "f1_delta_bdg_minus_communication":
            bdg_report.loc[cls,"f1-score"] - communication_report.loc[cls,"f1-score"]
    })

per_class_compare = pd.DataFrame(rows)
display(per_class_compare)
per_class_compare.to_csv(
    RESULT_DIR / "12_per_class_cg_vs_bdg.csv", index=False
)

In [ ]:

# CONFUSION MATRICES — ALL THREE METHODS (FIRST FINAL SEED)
def plot_and_save_cm(y_true, y_pred, title, stem):
    cm = confusion_matrix(
        y_true, y_pred,
        labels=np.arange(NUM_CLASSES)
    )
    fig, ax = plt.subplots(figsize=(12,10))
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
        ax=ax, xticks_rotation=45, values_format="d", colorbar=False
    )
    ax.set_title(title + " — Counts")
    plt.tight_layout()
    plt.savefig(RESULT_DIR / f"{stem}_counts.png", dpi=300, bbox_inches="tight")
    plt.show()

    cmn = confusion_matrix(
        y_true, y_pred,
        labels=np.arange(NUM_CLASSES),
        normalize="true"
    )
    fig, ax = plt.subplots(figsize=(12,10))
    ConfusionMatrixDisplay(cmn, display_labels=CLASS_NAMES).plot(
        ax=ax, xticks_rotation=45, values_format=".2f", colorbar=False
    )
    ax.set_title(title + " — Normalized")
    plt.tight_layout()
    plt.savefig(RESULT_DIR / f"{stem}_normalized.png", dpi=300, bbox_inches="tight")
    plt.show()

plot_and_save_cm(
    first_seed_details["communication_gat"]["y_true"],
    first_seed_details["communication_gat"]["y_pred"],
    "Communication Graph + GAT",
    "confusion_communication_gat"
)

plot_and_save_cm(
    first_seed_details["bdg_gat"]["y_true"],
    first_seed_details["bdg_gat"]["y_pred"],
    "Behavioral Dependency Graph + GAT",
    "confusion_bdg_gat"
)

plot_and_save_cm(
    baseline_first_seed_details["lightgbm"]["y_true"],
    baseline_first_seed_details["lightgbm"]["y_pred"],
    "Flow-level LightGBM",
    "confusion_lightgbm"
)


In [ ]:
# ============================================================
# FINAL RUNTIME / LATENCY TRADE-OFF ANALYSIS
# Supervisor request:
# Graph construction time + GAT inference time
# versus LightGBM inference time
# ============================================================

import time
import gc
import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

RUNTIME_REPEATS = 10
RUNTIME_WARMUPS = 2

# ------------------------------------------------------------
# DEVICE SYNCHRONIZATION
# ------------------------------------------------------------

def sync_device():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

# ------------------------------------------------------------
# LOCKED TEST DATA
# ------------------------------------------------------------

runtime_test_df = final_exp["test_df"].copy()
flow_pp = final_exp["flow_pp"]
bdd_pp = final_exp["bdd_pp"]

N_TEST_FLOWS = len(runtime_test_df)

# Precompute common node features outside runtime measurement.
# These are shared preprocessing operations and are not part of
# graph-construction overhead.

runtime_groups = []

for gid, g in runtime_test_df.groupby("_graph_id", sort=False):

    g = (
        g.sort_values(["timestamp", "_row_id"])
        .reset_index(drop=True)
    )

    X = flow_pp.transform_scaled(g)

    runtime_groups.append({
        "gid": gid,
        "gdf": g,
        "X": X
    })

N_TEST_GRAPHS = len(runtime_groups)

print("=" * 100)
print("RUNTIME / LATENCY ANALYSIS")
print("=" * 100)
print(f"Test flows:  {N_TEST_FLOWS:,}")
print(f"Test graphs: {N_TEST_GRAPHS:,}")
print(f"Window:      {BEST_WINDOW_SECONDS} seconds")
print(f"Device:      {DEVICE}")
print(f"Repeats:     {RUNTIME_REPEATS}")

# ------------------------------------------------------------
# COMMUNICATION GRAPH CONSTRUCTION
# ------------------------------------------------------------

def build_cg_runtime_graphs():

    graphs = []

    for item in runtime_groups:

        gid = item["gid"]
        g = item["gdf"]
        X = item["X"]

        pair_set = endpoint_candidate_pairs(
            g,
            CG_NEIGHBORS_PER_ENDPOINT
        )

        pairs = (
            np.asarray(sorted(pair_set), dtype=np.int64)
            if pair_set
            else np.empty((0, 2), dtype=np.int64)
        )

        if BEST_BDG_REPRESENTATION == "scalar":
            edge_features = scalar_cg_edge_attr(len(pairs))
        else:
            edge_features = vector_cg_edge_attr(len(pairs))

        edge_index, edge_attr = undirected_edges(
            pairs,
            edge_features
        )

        graph = Data(
            x=torch.tensor(X, dtype=torch.float32),
            edge_index=edge_index,
            edge_attr=edge_attr,
            y=torch.tensor(
                g["y"].values,
                dtype=torch.long
            )
        )

        graph.graph_id = str(gid)
        graph.n_nodes = len(g)
        graph.n_edges_undirected = len(pairs)

        graphs.append(graph)

    return graphs

# ------------------------------------------------------------
# CC-BDG CONSTRUCTION
# ------------------------------------------------------------

def build_bdg_runtime_graphs():

    graphs = []

    for item in runtime_groups:

        gid = item["gid"]
        g = item["gdf"]
        X = item["X"]

        # CC-BDG-specific behavioral processing is included
        # in the graph-construction timing.
        Z = bdd_pp.transform(g)

        pair_set = endpoint_candidate_pairs(
            g,
            CG_NEIGHBORS_PER_ENDPOINT
        )

        pairs = (
            np.asarray(sorted(pair_set), dtype=np.int64)
            if pair_set
            else np.empty((0, 2), dtype=np.int64)
        )

        C, S, P, B = compute_bdg_components(
            g,
            Z,
            pairs
        )

        if BEST_BDG_REPRESENTATION == "scalar":
            edge_features = scalar_bdg_edge_attr(
                C, S, P, B
            )
        else:
            edge_features = vector_bdg_edge_attr(
                C, S, P, B
            )

        edge_index, edge_attr = undirected_edges(
            pairs,
            edge_features
        )

        graph = Data(
            x=torch.tensor(X, dtype=torch.float32),
            edge_index=edge_index,
            edge_attr=edge_attr,
            y=torch.tensor(
                g["y"].values,
                dtype=torch.long
            )
        )

        graph.graph_id = str(gid)
        graph.n_nodes = len(g)
        graph.n_edges_undirected = len(pairs)

        graphs.append(graph)

    return graphs

# ------------------------------------------------------------
# GAT INFERENCE
# ------------------------------------------------------------

@torch.no_grad()
def timed_gat_predict(model, graphs):

    model.eval()

    loader = DataLoader(
        graphs,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    predictions = []

    for batch in loader:

        batch = batch.to(DEVICE)

        logits = model(
            batch.x,
            batch.edge_index,
            batch.edge_attr
        )

        pred = logits.argmax(dim=1)

        predictions.append(
            pred.detach().cpu().numpy()
        )

    return np.concatenate(predictions)

# ------------------------------------------------------------
# ALREADY-TRAINED MODELS
# ------------------------------------------------------------

cg_model = first_seed_details[
    "communication_gat"
]["model"]

bdg_model = first_seed_details[
    "bdg_gat"
]["model"]

lgbm_model = baseline_first_seed_details[
    "lightgbm"
]["model"]

X_test_lgbm = flow_baselines[
    "X_test_lgbm"
]

# ------------------------------------------------------------
# WARM-UP RUNS
# ------------------------------------------------------------

print("\nRunning warm-up...")

for _ in range(RUNTIME_WARMUPS):

    sync_device()
    _ = timed_gat_predict(
        cg_model,
        communication_sets["test"]
    )
    sync_device()

    _ = timed_gat_predict(
        bdg_model,
        bdg_sets["test"]
    )
    sync_device()

    _ = lgbm_model.predict(
        X_test_lgbm
    )

print("Warm-up complete.")

# ------------------------------------------------------------
# TIMED RUNS
# ------------------------------------------------------------

runtime_rows = []

for repeat in range(1, RUNTIME_REPEATS + 1):

    print(
        f"\nRuntime repeat "
        f"{repeat}/{RUNTIME_REPEATS}"
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ========================================================
    # CG-GAT
    # ========================================================

    t0 = time.perf_counter()

    cg_graphs = build_cg_runtime_graphs()

    t1 = time.perf_counter()

    cg_graph_time = t1 - t0

    sync_device()

    t2 = time.perf_counter()

    _ = timed_gat_predict(
        cg_model,
        cg_graphs
    )

    sync_device()

    t3 = time.perf_counter()

    cg_inference_time = t3 - t2
    cg_total_time = (
        cg_graph_time
        + cg_inference_time
    )

    runtime_rows.append({
        "repeat": repeat,
        "method": "CG-GAT",
        "graph_construction_s": cg_graph_time,
        "inference_s": cg_inference_time,
        "total_pipeline_s": cg_total_time
    })

    del cg_graphs

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ========================================================
    # CC-BDG-GAT
    # ========================================================

    t0 = time.perf_counter()

    bdg_graphs = build_bdg_runtime_graphs()

    t1 = time.perf_counter()

    bdg_graph_time = t1 - t0

    sync_device()

    t2 = time.perf_counter()

    _ = timed_gat_predict(
        bdg_model,
        bdg_graphs
    )

    sync_device()

    t3 = time.perf_counter()

    bdg_inference_time = t3 - t2
    bdg_total_time = (
        bdg_graph_time
        + bdg_inference_time
    )

    runtime_rows.append({
        "repeat": repeat,
        "method": "CC-BDG-GAT",
        "graph_construction_s": bdg_graph_time,
        "inference_s": bdg_inference_time,
        "total_pipeline_s": bdg_total_time
    })

    del bdg_graphs

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ========================================================
    # LIGHTGBM
    # ========================================================

    t0 = time.perf_counter()

    _ = lgbm_model.predict(
        X_test_lgbm
    )

    t1 = time.perf_counter()

    lgbm_inference_time = t1 - t0

    runtime_rows.append({
        "repeat": repeat,
        "method": "LightGBM",
        "graph_construction_s": 0.0,
        "inference_s": lgbm_inference_time,
        "total_pipeline_s": lgbm_inference_time
    })

    print(
        f"CG-GAT      | "
        f"Graph: {cg_graph_time:.4f}s | "
        f"Inference: {cg_inference_time:.4f}s | "
        f"Total: {cg_total_time:.4f}s"
    )

    print(
        f"CC-BDG-GAT  | "
        f"Graph: {bdg_graph_time:.4f}s | "
        f"Inference: {bdg_inference_time:.4f}s | "
        f"Total: {bdg_total_time:.4f}s"
    )

    print(
        f"LightGBM    | "
        f"Inference: {lgbm_inference_time:.4f}s"
    )

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

runtime_df = pd.DataFrame(runtime_rows)

runtime_summary = (
    runtime_df
    .groupby("method")
    .agg(
        graph_mean_s=(
            "graph_construction_s",
            "mean"
        ),
        graph_std_s=(
            "graph_construction_s",
            "std"
        ),
        inference_mean_s=(
            "inference_s",
            "mean"
        ),
        inference_std_s=(
            "inference_s",
            "std"
        ),
        total_mean_s=(
            "total_pipeline_s",
            "mean"
        ),
        total_std_s=(
            "total_pipeline_s",
            "std"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# ADD THROUGHPUT AND LATENCY
# ------------------------------------------------------------

runtime_summary[
    "throughput_flows_per_s"
] = (
    N_TEST_FLOWS
    / runtime_summary["total_mean_s"]
)

runtime_summary[
    "latency_ms_per_flow"
] = (
    runtime_summary["total_mean_s"]
    * 1000
    / N_TEST_FLOWS
)

# ------------------------------------------------------------
# ADD EXISTING 10-SEED PERFORMANCE
# ------------------------------------------------------------

performance_map = {
    "CG-GAT": "communication_gat",
    "CC-BDG-GAT": "bdg_gat",
    "LightGBM": "lightgbm"
}

macro_recall = []
macro_f1 = []

for method in runtime_summary["method"]:

    source_method = performance_map[method]

    row = all_method_summary[
        all_method_summary["method"]
        == source_method
    ].iloc[0]

    macro_recall.append(
        row["recall_macro_mean"]
    )

    macro_f1.append(
        row["f1_macro_mean"]
    )

runtime_summary[
    "macro_recall"
] = macro_recall

runtime_summary[
    "macro_f1"
] = macro_f1

# ------------------------------------------------------------
# LATENCY RELATIVE TO LIGHTGBM
# ------------------------------------------------------------

lgbm_latency = runtime_summary.loc[
    runtime_summary["method"] == "LightGBM",
    "total_mean_s"
].iloc[0]

runtime_summary[
    "latency_vs_lightgbm_x"
] = (
    runtime_summary["total_mean_s"]
    / lgbm_latency
)

# ------------------------------------------------------------
# FINAL SUPERVISOR-READY TABLE
# ------------------------------------------------------------

final_runtime_table = runtime_summary[
    [
        "method",
        "graph_mean_s",
        "graph_std_s",
        "inference_mean_s",
        "inference_std_s",
        "total_mean_s",
        "total_std_s",
        "throughput_flows_per_s",
        "latency_ms_per_flow",
        "macro_recall",
        "macro_f1",
        "latency_vs_lightgbm_x"
    ]
].copy()

final_runtime_table.columns = [
    "Method",
    "Graph Construction Mean (s)",
    "Graph Construction Std (s)",
    "Inference Mean (s)",
    "Inference Std (s)",
    "Total Pipeline Mean (s)",
    "Total Pipeline Std (s)",
    "Throughput (flows/s)",
    "Latency (ms/flow)",
    "Macro Recall",
    "Macro F1",
    "Latency vs LightGBM (x)"
]

print("\n")
print("=" * 120)
print("FINAL RECALL vs PIPELINE LATENCY TRADE-OFF")
print("=" * 120)

display(
    final_runtime_table.round({
        "Graph Construction Mean (s)": 4,
        "Graph Construction Std (s)": 4,
        "Inference Mean (s)": 4,
        "Inference Std (s)": 4,
        "Total Pipeline Mean (s)": 4,
        "Total Pipeline Std (s)": 4,
        "Throughput (flows/s)": 2,
        "Latency (ms/flow)": 4,
        "Macro Recall": 4,
        "Macro F1": 4,
        "Latency vs LightGBM (x)": 2
    })
)

# ------------------------------------------------------------
# DIRECT CC-BDG vs LIGHTGBM TRADE-OFF
# ------------------------------------------------------------

bdg_row = runtime_summary[
    runtime_summary["method"]
    == "CC-BDG-GAT"
].iloc[0]

cg_row = runtime_summary[
    runtime_summary["method"]
    == "CG-GAT"
].iloc[0]

lgbm_row = runtime_summary[
    runtime_summary["method"]
    == "LightGBM"
].iloc[0]

recall_gain_pp = (
    bdg_row["macro_recall"]
    - lgbm_row["macro_recall"]
) * 100

f1_difference_pp = (
    bdg_row["macro_f1"]
    - lgbm_row["macro_f1"]
) * 100

extra_latency_s = (
    bdg_row["total_mean_s"]
    - lgbm_row["total_mean_s"]
)

extra_bdg_graph_cost_s = (
    bdg_row["graph_mean_s"]
    - cg_row["graph_mean_s"]
)

print("\n")
print("=" * 120)
print("KEY SUPERVISOR TRADE-OFF")
print("=" * 120)

print(
    f"LightGBM inference: "
    f"{lgbm_row['inference_mean_s']:.4f} "
    f"± {lgbm_row['inference_std_s']:.4f} s"
)

print(
    f"CG-GAT graph construction: "
    f"{cg_row['graph_mean_s']:.4f} "
    f"± {cg_row['graph_std_s']:.4f} s"
)

print(
    f"CG-GAT inference: "
    f"{cg_row['inference_mean_s']:.4f} "
    f"± {cg_row['inference_std_s']:.4f} s"
)

print(
    f"CG-GAT total pipeline: "
    f"{cg_row['total_mean_s']:.4f} "
    f"± {cg_row['total_std_s']:.4f} s"
)

print(
    f"CC-BDG graph construction: "
    f"{bdg_row['graph_mean_s']:.4f} "
    f"± {bdg_row['graph_std_s']:.4f} s"
)

print(
    f"CC-BDG GAT inference: "
    f"{bdg_row['inference_mean_s']:.4f} "
    f"± {bdg_row['inference_std_s']:.4f} s"
)

print(
    f"CC-BDG total pipeline: "
    f"{bdg_row['total_mean_s']:.4f} "
    f"± {bdg_row['total_std_s']:.4f} s"
)

print(
    f"\nCC-BDG Macro Recall: "
    f"{bdg_row['macro_recall']:.4f}"
)

print(
    f"LightGBM Macro Recall: "
    f"{lgbm_row['macro_recall']:.4f}"
)

print(
    f"Macro-Recall gain of CC-BDG over LightGBM: "
    f"{recall_gain_pp:.2f} percentage points"
)

print(
    f"Macro-F1 difference of CC-BDG vs LightGBM: "
    f"{f1_difference_pp:+.2f} percentage points"
)

print(
    f"Additional CC-BDG pipeline latency vs LightGBM: "
    f"{extra_latency_s:.4f} s"
)

print(
    f"CC-BDG pipeline latency relative to LightGBM: "
    f"{bdg_row['latency_vs_lightgbm_x']:.2f}x"
)

print(
    f"Additional CC-BDG graph-construction cost vs CG: "
    f"{extra_bdg_graph_cost_s:.4f} s"
)

print(
    f"CC-BDG throughput: "
    f"{bdg_row['throughput_flows_per_s']:,.2f} flows/s"
)

print(
    f"LightGBM throughput: "
    f"{lgbm_row['throughput_flows_per_s']:,.2f} flows/s"
)

# ------------------------------------------------------------
# SAVE RESULTS
# ------------------------------------------------------------

runtime_df.to_csv(
    RESULT_DIR / "runtime_raw_repeats.csv",
    index=False
)

final_runtime_table.to_csv(
    RESULT_DIR / "runtime_recall_latency_tradeoff.csv",
    index=False
)

print("\nSaved:")
print(
    RESULT_DIR / "runtime_raw_repeats.csv"
)
print(
    RESULT_DIR / "runtime_recall_latency_tradeoff.csv"
)

In [ ]:

# CELL 17 — TRAINING CURVES
for method, title in [
    ("communication_gat", "Communication Graph + GAT"),
    ("bdg_gat", "Behavioral Dependency Graph + GAT")
]:
    h = first_seed_details[method]["history"]

    plt.figure(figsize=(8,5))
    plt.plot(h["epoch"], h["train_loss"])
    plt.xlabel("Epoch")
    plt.ylabel("Training loss")
    plt.title(title + " — Training Loss")
    plt.grid(alpha=.25)
    plt.tight_layout()
    plt.savefig(RESULT_DIR / f"training_loss_{method}.png", dpi=300)
    plt.show()

    plt.figure(figsize=(8,5))
    plt.plot(h["epoch"], h["val_f1_macro"], label="Macro-F1")
    plt.plot(h["epoch"], h["val_f1_weighted"], label="Weighted-F1")
    plt.xlabel("Epoch")
    plt.ylabel("Validation F1")
    plt.title(title + " — Validation F1")
    plt.legend()
    plt.grid(alpha=.25)
    plt.tight_layout()
    plt.savefig(RESULT_DIR / f"validation_f1_{method}.png", dpi=300)
    plt.show()



# How to interpret the final paper result

The controlled comparison is:

- **CG:** shared-endpoint topology with uniform edge attribute `1.0`;
- **BDG:** the exact same shared-endpoint topology with behavioral edge attribute `(1 + S + P + B) / 4`.

Both methods use the same dataset, split, temporal window, node features, edge-aware GAT architecture, optimizer, loss, and training protocol.

No BDG-specific parameter search or edge filtering occurs. Therefore, any performance difference is attributable to the behavioral edge representation rather than asymmetric model tuning.


# Final comparison interpretation

The final study contains three classifiers evaluated on the same locked split:

1. **CG-GAT** — flow nodes with shared-endpoint communication topology and uniform edge attribute.
2. **CC-BDG-GAT** — identical flow nodes and topology, with behavioral edge attributes derived from communication, service, protocol, and traffic pattern.
3. **LightGBM** — flow-only tree baseline using the same imputed numerical features, without graph relationships.

The temporal-window ablation remains restricted to **CG-GAT and CC-BDG-GAT**, because temporal window is a graph-construction choice. LightGBM is introduced only after the common graph window and graph-isolated split are locked.

This preserves two separate comparisons:

- **Controlled graph-semantic comparison:** CG-GAT vs CC-BDG-GAT.
- **Broader predictive baseline comparison:** CG-GAT / CC-BDG-GAT vs LightGBM.



# Final architecture protocol

Only the GAT architecture is under study here.

Unchanged:
- CC-BDG remains `[C,S,P,B]`;
- no TCP-state dependency is used;
- graph construction and topology are unchanged;
- split logic is unchanged;
- temporal-window handling is unchanged;
- LightGBM baseline is unchanged.

The best GAT architecture is selected using CC-BDG validation Macro-F1 and
then applied identically to both CG and CC-BDG for the final test.
